### ============================================================
### DATA COLLECTION NOTEBOOK — DO NOT RE-RUN
### Outputs are already saved to /data/. Re-running requires API connection and will overwrite archived raw data.
### ============================================================

# 2
In this notebook I collect the members data and apply a gender guesser. 

In [18]:
import requests
import pandas as pd
from time import sleep

all_members = []
page = 0
base_url = "https://api.oireachtas.ie/v1/members"

while True:
    url = f"{base_url}?limit=100&skip={page * 100}"
    print(f"Fetching page {page + 1}: {url}")
    response = requests.get(url)
    if response.status_code != 200:
        print(f"Error fetching page {page+1}: {response.status_code}")
        break

    data = response.json()
    results = data.get("results", [])
    if not results:
        print("No more results, stopping.")
        break

    for item in results:
        member = item.get("member", {})
        fullName = member.get("fullName")
        firstName = member.get("firstName")
        lastName = member.get("lastName")
        gender = member.get("gender")

        memberships = member.get("memberships", [])
        if memberships:
            for m in memberships:
                membership = m["membership"]
                house = membership.get("house", {}).get("showAs")
                
                constituency = None
                if membership.get("represents"):
                    rep = membership["represents"][0].get("represent", {})
                    constituency = rep.get("showAs")

                party = None
                if membership.get("parties"):
                    p = membership["parties"][0].get("party", {})
                    party = p.get("showAs")

                offices = []
                for o in membership.get("offices", []):
                    office = o.get("office", {})
                    if office.get("showAs"):
                        offices.append(office["showAs"])
                role = offices[0] if offices else "TD or Senator"

                start_date = membership.get("dateRange", {}).get("start")
                end_date = membership.get("dateRange", {}).get("end")

                all_members.append({
                    "fullName": fullName,
                    "firstName": firstName,
                    "lastName": lastName,
                    "gender": gender,
                    "house": house,
                    "constituency": constituency,
                    "party": party,
                    "role": role,
                    "uri": member.get("uri"),
                    "membership_start": start_date,
                    "membership_end": end_date
                })
        else:
            all_members.append({
                "fullName": fullName,
                "firstName": firstName,
                "lastName": lastName,
                "gender": gender,
                "house": None,
                "constituency": None,
                "party": None,
                "role": None,
                "uri": member.get("uri"),
                "membership_start": None,
                "membership_end": None
            })

    print(f"Page {page + 1}: {len(results)} members processed")
    page += 1
    sleep(0.2)

    if page > 30:  # safety stop
        print("Reached 30 pages, stopping.")
        break

# Convert to DataFrame
df_members_flat = pd.DataFrame(all_members)
print(f"\nTotal members collected: {len(df_members_flat)}")


Fetching page 1: https://api.oireachtas.ie/v1/members?limit=100&skip=0
✅ Page 1: 100 members processed
Fetching page 2: https://api.oireachtas.ie/v1/members?limit=100&skip=100
✅ Page 2: 100 members processed
Fetching page 3: https://api.oireachtas.ie/v1/members?limit=100&skip=200
✅ Page 3: 100 members processed
Fetching page 4: https://api.oireachtas.ie/v1/members?limit=100&skip=300
✅ Page 4: 100 members processed
Fetching page 5: https://api.oireachtas.ie/v1/members?limit=100&skip=400
✅ Page 5: 100 members processed
Fetching page 6: https://api.oireachtas.ie/v1/members?limit=100&skip=500
✅ Page 6: 100 members processed
Fetching page 7: https://api.oireachtas.ie/v1/members?limit=100&skip=600
✅ Page 7: 100 members processed
Fetching page 8: https://api.oireachtas.ie/v1/members?limit=100&skip=700
✅ Page 8: 100 members processed
Fetching page 9: https://api.oireachtas.ie/v1/members?limit=100&skip=800
✅ Page 9: 100 members processed
Fetching page 10: https://api.oireachtas.ie/v1/members?li

In [19]:
df_members_flat.head(-10)

,fullName,firstName,lastName,gender,house,constituency,party,role,uri,membership_start,membership_end
0,Henry J. J. Abbott,Henry J. J.,Abbott,,25th Dáil,Longford-Westmeath,Fianna Fáil,TD or Senator,https://data.oireachtas.ie/ie/oireachtas/membe...,1987-03-10,1989-05-25
1,Caroline Acheson,Caroline,Acheson,,22nd Dáil,Tipperary South,Fianna Fáil,TD or Senator,https://data.oireachtas.ie/ie/oireachtas/membe...,1981-06-30,1982-02-27
2,Gerry Adams,Gerry,Adams,,32nd Dáil,Louth,Sinn Féin,TD or Senator,https://data.oireachtas.ie/ie/oireachtas/membe...,2016-03-10,2020-01-14
3,Gerry Adams,Gerry,Adams,,31st Dáil,Louth,Sinn Féin,TD or Senator,https://data.oireachtas.ie/ie/oireachtas/membe...,2011-03-09,2016-03-09
4,Patrick Agnew,Patrick,Agnew,,22nd Dáil,Louth,Independent,TD or Senator,https://data.oireachtas.ie/ie/oireachtas/membe...,1981-06-30,1982-02-27
...,...,...,...,...,...,...,...,...,...,...,...
7205,Aodhán Ó Ríordáin,Aodhán,Ó Ríordáin,,31st Dáil,Dublin North-Central,Labour Party,TD or Senator,https://data.oireachtas.ie/ie/oireachtas/membe...,2011-03-09,2016-03-09
7206,Aodhán Ó Ríordáin,Aodhán,Ó Ríordáin,,33rd Dáil,Dublin Bay North,Labour Party,TD or Senator,https://data.oireachtas.ie/ie/oireachtas/membe...,2020-02-08,2024-07-15
7207,Pádraig Ó Siochfhradha,Pádraig,Ó Siochfhradha,,10th Seanad,Nominated by the Taoiseach,Independent,TD or Senator,https://data.oireachtas.ie/ie/oireachtas/membe...,1961-12-14,1964-11-19
7208,Pádraig Ó Siochfhradha,Pádraig,Ó Siochfhradha,,9th Seanad,Nominated by the Taoiseach,Independent,TD or Senator,https://data.oireachtas.ie/ie/oireachtas/membe...,1957-05-22,1961-09-01


In [21]:
# Save as CSV
df_members_flat.to_csv("data/members.csv", index=False)


In [24]:
# gender guesser

import pandas as pd
import gender_guesser.detector as gender

# Initialize the gender detector
d = gender.Detector()

# Load your members CSV
members = pd.read_csv("data/members.csv")

# Function to infer gender if empty
def guess_gender(row):
    if pd.isna(row['gender']) or row['gender'].strip() == "":
        first_name = row['firstName']
        if pd.notna(first_name):
            clean_name = first_name.split()[0].split(".")[0]  # take first "real" name
            g = d.get_gender(clean_name)
            if g in ["male", "mostly_male"]:
                return "male"
            elif g in ["female", "mostly_female"]:
                return "female"
            else:
                return "unknown"
        else:
            return "unknown"
    else:
        return row['gender']


# Apply to the DataFrame
members['gender_guess'] = members.apply(guess_gender, axis=1)

# save to new CSV
members.to_csv("data/members_with_gender_guess.csv", index=False)

# Quick check
members[['firstName', 'fullName', 'gender', 'gender_guess']].head(20)


,firstName,fullName,gender,gender_guess
0,Henry J. J.,Henry J. J. Abbott,NaN,male
1,Caroline,Caroline Acheson,NaN,female
2,Gerry,Gerry Adams,NaN,male
3,Gerry,Gerry Adams,NaN,male
4,Patrick,Patrick Agnew,NaN,male
5,Garret,Garret Ahearn,NaN,male
6,Garret,Garret Ahearn,NaN,male
7,Theresa,Theresa Ahearn,NaN,female
8,Theresa,Theresa Ahearn,NaN,female
9,Theresa,Theresa Ahearn,NaN,female


In [25]:
members['gender_guess'].value_counts()

gender_guess
male       6136
female      647
unknown     437
Name: count, dtype: int64

In [26]:
members.head()

,fullName,firstName,lastName,gender,house,constituency,party,role,uri,membership_start,membership_end,gender_guess
0,Henry J. J. Abbott,Henry J. J.,Abbott,NaN,25th Dáil,Longford-Westmeath,Fianna Fáil,TD or Senator,https://data.oireachtas.ie/ie/oireachtas/membe...,1987-03-10,1989-05-25,male
1,Caroline Acheson,Caroline,Acheson,NaN,22nd Dáil,Tipperary South,Fianna Fáil,TD or Senator,https://data.oireachtas.ie/ie/oireachtas/membe...,1981-06-30,1982-02-27,female
2,Gerry Adams,Gerry,Adams,NaN,32nd Dáil,Louth,Sinn Féin,TD or Senator,https://data.oireachtas.ie/ie/oireachtas/membe...,2016-03-10,2020-01-14,male
3,Gerry Adams,Gerry,Adams,NaN,31st Dáil,Louth,Sinn Féin,TD or Senator,https://data.oireachtas.ie/ie/oireachtas/membe...,2011-03-09,2016-03-09,male
4,Patrick Agnew,Patrick,Agnew,NaN,22nd Dáil,Louth,Independent,TD or Senator,https://data.oireachtas.ie/ie/oireachtas/membe...,1981-06-30,1982-02-27,male
